# Enhanced Market Basket Analysis

This notebook modernizes the original `Association Analysis.ipynb` workflow.

**What changed vs the original notebook**
1. Uses true **Apriori** (level-wise candidate generation + support pruning) instead of brute-force `combinations` on orders of *exactly* size 4
2. Adds **FP-Growth** as a complementary, typically faster algorithm
3. Mines **association rules** with confidence, lift, leverage, and conviction
4. Adds a **pairwise PMI / Jaccard** affinity view and a co-occurrence network
5. Produces publication-ready figures under `figures/`

Default data is a planted Instacart-like sample (`data/sample/`). Drop real Instacart CSVs into `data/raw/` to analyze the full dataset.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('.').resolve()
for candidate in [Path('.').resolve(), Path('..').resolve()]:
    if (candidate / 'src' / 'mba').exists():
        ROOT = candidate
        break

sys.path.insert(0, str(ROOT / 'src'))

from mba.affinity import pairwise_pmi
from mba.apriori import apriori, frequent_itemsets_to_frame
from mba.data import (
    filter_frequent_products,
    filter_orders_by_size,
    load_order_products,
    load_products,
    product_name_map,
    transactions_from_orders,
)
from mba.fp_growth import fp_growth
from mba.rules import association_rules
from mba.visualize import (
    plot_cooccurrence_network,
    plot_itemset_support,
    plot_metric_comparison,
    plot_pmi_heatmap_pairs,
    plot_rules_scatter,
    plot_top_products,
)

pd.set_option('display.max_colwidth', 120)
print('Project root:', ROOT)

## 1. Load & preprocess

Prefer `data/raw/` (real Instacart) when present; otherwise use the sample generator output.

In [ ]:
candidates = [ROOT / 'data' / 'raw', ROOT / 'data' / 'sample', ROOT]
op_path = pr_path = None
for base in candidates:
    if (base / 'order_products__train.csv').exists() and (base / 'products.csv').exists():
        op_path, pr_path = base / 'order_products__train.csv', base / 'products.csv'
        break
assert op_path is not None, 'Run: python scripts/generate_sample_data.py'

order_products = load_order_products(op_path)
products = load_products(pr_path)
name_map = product_name_map(products)

print(f'Orders file: {op_path}')
print(f'Rows: {len(order_products):,} | Orders: {order_products.order_id.nunique():,} | Products: {order_products.product_id.nunique():,}')
order_products.head()

In [ ]:
# Keep reasonably frequent products and baskets with at least 2 items
# (original notebook incorrectly filtered to orders of *exactly* size 4)
MIN_COUNT = 40
MIN_SUPPORT = 0.008
MAX_LEN = 4

compacted, kept_ids = filter_frequent_products(order_products, min_count=MIN_COUNT)
compacted = filter_orders_by_size(compacted, min_size=2)
transactions = transactions_from_orders(compacted)

print(f'Kept products: {len(kept_ids):,}')
print(f'Baskets: {len(transactions):,}')
print(f'Mean basket size: {sum(len(t) for t in transactions)/len(transactions):.2f}')

## 2. Exploratory view

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_top_products(compacted, name_map, top_n=15, ax=ax)
plt.show()

## 3. Apriori (enhanced)

Level-wise mining with the anti-monotonicity prune: if an itemset is infrequent, none of its supersets can be frequent.

In [ ]:
freq_apriori = apriori(transactions, min_support=MIN_SUPPORT, max_len=MAX_LEN)
apriori_df = frequent_itemsets_to_frame(freq_apriori, name_map=name_map)
print(f'Frequent itemsets: {len(apriori_df):,}')
apriori_df.groupby('size').size().rename('count').to_frame()

In [ ]:
# Original project goal: top collections of four items
fours = apriori_df[apriori_df['size'] == 4].head(10)
fours

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_itemset_support(apriori_df, size=4, top_n=8, ax=ax)
plt.show()

## 4. FP-Growth (complementary)

Same frequent-itemset output shape, usually faster because it compresses baskets into an FP-tree and avoids candidate generation.

In [ ]:
freq_fp = fp_growth(transactions, min_support=MIN_SUPPORT, max_len=MAX_LEN)
fp_df = frequent_itemsets_to_frame(freq_fp, name_map=name_map)

print('Same itemsets?', set(freq_apriori) == set(freq_fp))
fig, ax = plt.subplots(figsize=(8, 5))
plot_metric_comparison(apriori_df, fp_df, ax=ax)
plt.show()

## 5. Association rules

Rules turn frequent itemsets into actionable *if → then* statements ranked by lift (how much more often the consequent appears given the antecedent than by chance).

In [ ]:
rules = association_rules(
    freq_apriori,
    min_confidence=0.2,
    min_lift=1.05,
    name_map=name_map,
)
print(f'Rules: {len(rules):,}')
rules.head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_rules_scatter(rules, top_n=50, ax=ax)
plt.show()

## 6. Pairwise PMI affinity (complementary technique)

Support-based mining prefers popular items. **PMI** highlights pairs that co-occur *more than chance*, which surfaces niche but strong affinities.

In [ ]:
pmi = pairwise_pmi(transactions, min_count=8, top_n=40, name_map=name_map)
pmi.head(12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_pmi_heatmap_pairs(pmi, top_n=12, ax=axes[0])
plot_cooccurrence_network(pmi, top_edges=22, ax=axes[1])
plt.tight_layout()
plt.show()

## Takeaways

| Technique | Strength | Use when |
|---|---|---|
| **Apriori** | Interpretable, exact under thresholds | Teaching / debugging support prune |
| **FP-Growth** | Faster on denser data | Production frequent-itemset mining |
| **Association rules** | Actionable recommendations | Cross-sell / shelf placement |
| **PMI / network** | Finds surprising affinities | Cold-start bundles beyond popular SKUs |

On the sample data, the same two 4-item themes from the original notebook resurface first: **sparkling waters** and **berry mixes** — now with proper support pruning, rule metrics, and visuals.